In [ ]:
TRAIN_PATH = '/content/drive/MyDrive/DeepX Hackathon 2026/DeepX dataSet/data_for_aspect/aspect_train_df.csv'
VAL_PATH = '/content/drive/MyDrive/DeepX Hackathon 2026/DeepX dataSet/data_for_aspect/aspect_val_df.csv'

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from scipy.special import expit
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

from transformers import EarlyStoppingCallback
from sklearn.metrics import f1_score, precision_score, recall_score

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

In [ ]:
train_df.head()

,id,original_text,text_for_aspect,food,service,price,cleanliness,delivery,ambiance,app_experience,general,none
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,لا يوجد الدفع بالبطاقه عند الاستلام,0,0,0,0,1,0,1,0,0
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,0,1,0,1,0,1,0,0,0
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,تجربة سيية سالتهم الاكل هياخد وقت قد ايه قالول...,1,1,0,0,1,0,0,0,0
3,3024,احلي مكان فزايد,احلي مكان فزايد,0,0,0,0,0,0,0,1,0
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,الفطير حلو جدا الاحجام تحفة بالنسبه للسعر فا ي...,1,0,1,0,0,0,0,0,0


In [ ]:
val_df.head()

,id,original_text,text_for_aspect,food,service,price,cleanliness,delivery,ambiance,app_experience,general,none
0,4446,مريم سوتلي الاظافررر تحفههه اوييي ❤️❤️❤️❤️❤️,مريم سوتلي الاظافرر تحفهه اويي ❤️❤️❤️❤️❤️,0,1,0,0,0,0,0,0,0
1,8612,التطبيق جميل .. أتمنى إضافة البحث عن طريق الخر...,التطبيق جميل .. اتمني اضافة البحث عن طريق الخر...,0,0,0,0,0,0,1,0,0
2,6729,سراقين مكتوب وصلت السياره والسواق مارضى يقول و...,سراقين مكتوب وصلت السياره والسواق مارضي يقول و...,0,1,1,0,1,0,0,0,0
3,6292,سي جيدا,سي جيدا,0,0,0,0,0,0,0,1,0
4,1639,مكان ممتاز جدا و الخدمة جيده جدا,مكان ممتاز جدا و الخدمة جيده جدا,0,1,0,0,0,1,0,0,0


In [ ]:
aspect_cols = [
    "food",
    "service",
    "price",
    "cleanliness",
    "delivery",
    "ambiance",
    "app_experience",
    "general",
    "none"
]

model_name = "xlm-roberta-large"

In [ ]:
train_data = train_df[["text_for_aspect"] + aspect_cols].copy()
val_data   = val_df[["text_for_aspect"] + aspect_cols].copy()

train_data["text_for_aspect"] = train_data["text_for_aspect"].astype(str)
val_data["text_for_aspect"] = val_data["text_for_aspect"].astype(str)

train_data[aspect_cols] = train_data[aspect_cols].astype(float)
val_data[aspect_cols] = val_data[aspect_cols].astype(float)

train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

In [ ]:
print(train_dataset)
print(val_dataset)

Dataset({
    features: ['text_for_aspect', 'food', 'service', 'price', 'cleanliness', 'delivery', 'ambiance', 'app_experience', 'general', 'none'],
    num_rows: 1971
})
Dataset({
    features: ['text_for_aspect', 'food', 'service', 'price', 'cleanliness', 'delivery', 'ambiance', 'app_experience', 'general', 'none'],
    num_rows: 500
})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_aspect_data(example):
    encoding = tokenizer(
        example["text_for_aspect"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    encoding["labels"] = [float(example[col]) for col in aspect_cols]
    return encoding

train_dataset = train_dataset.map(tokenize_aspect_data)
val_dataset = val_dataset.map(tokenize_aspect_data)

cols_to_keep = ["input_ids", "attention_mask", "labels"]
train_dataset.set_format(type="torch", columns=cols_to_keep)
val_dataset.set_format(type="torch", columns=cols_to_keep)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1971 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(aspect_cols),
    problem_type="multi_label_classification"
)

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = expit(logits)

    threshold = 0.35
    preds = (probs >= threshold).astype(int)

    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(labels, preds),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./aspect_model_train",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=5e-5,
    num_train_epochs=15,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    weight_decay=0.01,

    warmup_ratio=0.1,
    lr_scheduler_type="linear",

    fp16=True,

    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="micro_recall",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.0
        )
    ]
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
train = trainer.train()

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.404453,0.336292,0.580429,0.319046,0.664110,0.515476,0.139111,2.562200,195.144000,6.245000
2,0.284154,0.243232,0.744081,0.623606,0.706638,0.785714,0.100889,3.145600,158.953000,5.086000
3,0.217496,0.197010,0.801653,0.680956,0.795082,0.808333,0.074667,2.868800,174.286000,5.577000
4,0.176615,0.205590,0.800905,0.692801,0.762931,0.842857,0.078222,2.900600,172.380000,5.516000
5,0.131434,0.183626,0.821553,0.726638,0.800226,0.844048,0.068444,2.912500,171.677000,5.494000
6,0.109888,0.187647,0.825075,0.767551,0.827545,0.822619,0.065111,2.936100,170.294000,5.449000
7,0.081625,0.172600,0.838860,0.777576,0.820250,0.858333,0.061556,2.918000,171.353000,5.483000
8,0.065676,0.168258,0.854631,0.794361,0.841801,0.867857,0.055111,2.967600,168.486000,5.392000
9,0.051096,0.200277,0.842167,0.773205,0.833333,0.851190,0.059556,2.920000,171.234000,5.479000
10,0.034069,0.185972,0.859954,0.808658,0.836712,0.884524,0.053778,2.872900,174.039000,5.569000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [ ]:
best_thresholds = {
    "food": 0.57,
    "service": 0.44,
    "price": 0.35,
    "cleanliness": 0.21,
    "delivery": 0.18,
    "ambiance": 0.40,
    "app_experience": 0.39,
    "general": 0.63,
    "none": 0.10
}

In [ ]:
trainer.save_model("./best_aspect_model")
tokenizer.save_pretrained("./best_aspect_model")
import json

with open("./best_aspect_model/thresholds.json", "w") as f:
    json.dump(best_thresholds, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import shutil
import os

destination_path = "/content/drive/MyDrive/DeepX Hackathon 2026/models/"

# Remove the directory if it already exists to ensure a clean copy
if os.path.exists(destination_path):
    shutil.rmtree(destination_path)

shutil.copytree(
    "/content/best_aspect_model",
    destination_path
)

'/content/drive/MyDrive/DeepX Hackathon 2026/models/'

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import json

model_path = '/content/drive/MyDrive/DeepX Hackathon 2026/models'

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

with open(f"{model_path}/thresholds.json") as f:
    thresholds = json.load(f)

model.eval()

def predict_aspects(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits)[0].cpu().numpy()

    aspects = []

    for i, aspect in enumerate(aspect_cols):
        if aspect == "none":
            continue

        if probs[i] >= thresholds[aspect]:
            aspects.append(aspect)

    if len(aspects) == 0:
        aspects = ["none"]

    return aspects

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
predict_aspects(''' تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالولي نص ساعة فعد ساعة ونص
الاكل يعني كويس ولكن كمية قليلة''')

['food', 'service', 'delivery']